# CV Masterclass 06: OCR & Document AI

Welcome to Notebook 15. Until now, we have treated images as collections of objects (Detection) or pixels (Segmentation). Today, we bridge the gap between **Vision** and **Language**. 

Optical Character Recognition (OCR) is the art of teaching machines to 'read'. It is not just about identifying a 'letter'; it is about understanding spatial sequences, geometry, and semantics. In this module, we move from simple Tesseract-style pipelines to modern **Document AI** where models parse structured information directly from pixels.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"In traditional binarization, we use a hard threshold (e.g., 0.5). Why does DBNet's use of a 'differentiable step function' ($\hat{B} = \frac{1}{1 + e^{-k(P-T)}}$) allow the network to learn the threshold itself, and how does this affect the robustness of bounding box tight-fitting during training?"*

> *"CTC Loss uses a 'blank' character to handle unaligned sequences. If we have a sequence of 50 feature frames predicting 'CAT' (3 characters), how does the forward-backward algorithm ensure that 'C-A-T', 'CCAA-TT', and '---C-AA-T' all collapse to the same ground truth, and what happens to the gradient flow if the prediction length is shorter than the target?"*

---

## Table of Contents
1. **The OCR Pipeline:** Detection + Recognition.
2. **Text Detection:** DBNet (Differentiable Binarization) and CRAFT (Character Region Awareness).
3. **Text Recognition:** CRNN (Convolutional Recurrent Neural Networks) and the absolute necessity of **CTC Loss**.
4. **Transformer-based End-to-End OCR:** TrOCR and Donut (Document Understanding Transformer).
5. **Common Pitfalls:** Perspective, noise, and script complexity.


## 1. The OCR Pipeline: Detection + Recognition

Modern OCR is almost always a two-stage process (unless using an E2E Transformer).

### Stage 1: Text Detection
The model finds the bounding boxes or polygons containing text. Unlike object detection (where boxes are often square/rectangular), text detection must handle **arbitrary shapes**, curved lines, and extreme aspect ratios (long sentences).

### Stage 2: Text Recognition
Once a crop of text is extracted, it is passed to a second network that converts the image patch into a string of characters. This is a **Sequence-to-Sequence** problem.

| Feature | Detection | Recognition |
|---|---|---|
| **Goal** | Locate 'where' text is. | Identify 'what' the text says. |
| **Output** | Polygons/Bounding Boxes. | Character Strings. |
| **Architectures** | DBNet, CRAFT, PANet. | CRNN, Rosetta, TrOCR. |

## 2. Text Detection: DBNet & CRAFT

### DBNet: Real-time Differentiable Binarization
Standard segmentation networks output a probability map. To get boxes, we usually apply a hard threshold (0.3 or 0.5). However, this hard thresholding is **not differentiable**, meaning the network cannot learn how to optimize the thresholding process itself.

**The Innovation:** DBNet introduces a **Differentiable Binarization (DB)** function:
$$ \hat{B}_{i,j} = \frac{1}{1 + e^{-k(P_{i,j} - T_{i,j})}} $$

Where:
- $P$: The probability map (predicted by the CNN).
- $T$: The threshold map (also predicted by the CNN!).
- $k$: A steepness factor (typically 50).

As $k \to \infty$, the function approximates a step function. Because it is differentiable, the network can learn a variable threshold map $T$ that adaptively separates text from background, even in challenging lighting conditions.

### CRAFT: Character Region Awareness
Instead of detecting words or lines, CRAFT detects individual **character regions** and the **links** between them. This allows it to handle curved text by essentially 'stitching' characters together based on affinity scores.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

class DifferentiableBinarization(nn.Module):
    """
    Implements the DB function from the DBNet paper.
    It allows the network to learn an adaptive threshold map.
    """
    def __init__(self, k=50):
        super().__init__()
        self.k = k

    def forward(self, prob_map, threshold_map):
        # Formula: 1 / (1 + exp(-k * (P - T)))
        # This creates a 'soft' step function around the threshold T
        binary_map = torch.reciprocal(1 + torch.exp(-self.k * (prob_map - threshold_map)))
        return binary_map

# TEST IT
db_layer = DifferentiableBinarization(k=50)

# Simulate a probability map [0, 1]
p = torch.linspace(0, 1, 100)
# Simulate a target threshold of 0.4
t = torch.full_like(p, 0.4)

b = db_layer(p, t)

plt.figure(figsize=(10, 4))
plt.plot(p.numpy(), b.detach().numpy(), label='Differentiable Binarization (k=50)')
plt.axvline(0.4, color='red', linestyle='--', label='Learnable Threshold T=0.4')
plt.title("DBNet: The Differentiable Step Function")
plt.xlabel("Probability P")
plt.ylabel("Binary Output B")
plt.legend()
plt.grid(True)
plt.show()

assert b[40] > 0.4 and b[40] < 0.6  # Smooth transition at threshold
assert b[0] < 0.01 and b[-1] > 0.99 # Saturates at extremes

## 3. Text Recognition: CRNN & CTC Loss

The golden standard for text recognition is the **CRNN (Convolutional Recurrent Neural Network)**.

1.  **CNN:** Extracts a feature map from the image. We take the 'slices' of this feature map along the width as a sequence.
2.  **RNN (LSTM/GRU):** Processes the sequence of slices to understand spatial context (e.g., 'is this vertical stroke part of an H or an I?').
3.  **Linear:** Predicts character probabilities for each slice.

### The Problem: Variable Length Alignment
If our image is 100 pixels wide and we have 25 feature slices, but the word is "CAT" (3 characters), how do we supervise the model? We don't know exactly which slice corresponds to which character.

### The Solution: Connectionist Temporal Classification (CTC) Loss
CTC allows us to train without alignment by introducing a special **[blank]** token (represented as $\epsilon$).

**The Rules of CTC Decoding:**
1.  **Collapse Repeats:** `CC-AA-TT` $\to$ `CAT`
2.  **Remove Blanks:** `C---A---T` $\to$ `CAT`
3.  **Blanks separate repeats:** `CLL` (collapsed) vs `C L - L` (becomes 'CLL' because the blank prevents the two Ls from merging).

CTC iterates over all possible paths that collapse to the target string and maximizes their log-probability sum.

In [ ]:
def ctc_decode_simple(sequence, blank=0):
    """
    A simple greedy CTC decoder logic.
    1. Collapse adjacent duplicates.
    2. Remove blanks.
    """
    # 1. Collapse adjacent duplicates
    collapsed = []
    prev = None
    for char in sequence:
        if char != prev:
            collapsed.append(char)
        prev = char
    
    # 2. Remove blanks
    final = [char for char in collapsed if char != blank]
    return final

# TEST IT
# Map: 0: '-', 1: 'C', 2: 'A', 3: 'T'
sample_output = [1, 1, 0, 0, 2, 2, 2, 0, 3, 3, 3, 0] # Predicted path: CC--AAATTT-
decoded = ctc_decode_simple(sample_output)
print(f"Raw Path IDs: {sample_output}")
print(f"Decoded Token IDs: {decoded}")

assert decoded == [1, 2, 3] # Should collapse to 'CAT'

# Case: Repeated characters separated by blank
repeats_with_blank = [1, 1, 0, 1, 1] # CC-CC
decoded_repeat = ctc_decode_simple(repeats_with_blank)
print(f"Repeated 'C' separated by blank: {decoded_repeat}")
assert decoded_repeat == [1, 1] # Should be 'CC' (like in 'OCCUR')


## 4. Transformer-based End-to-End OCR: TrOCR & Donut

The era of CNN+RNN+CTC is being challenged by **Generative OCR**.

### TrOCR (Transformer OCR)
TrOCR does away with the complex pipeline. It uses an **Encoder-Decoder Transformer** architecture:
- **Encoder:** A Vision Transformer (ViT) that processes the image into patches.
- **Decoder:** A text transformer (like GPT-2) that attends to the visual features and generates text autoregressively.

**Why?** It removes the need for CTC alignment and handles complex layout geometries more naturally.

### Donut (Document Understanding Transformer)
Donut takes it a step further: **Reading without OCR**. 
Instead of extracting boxes $\to$ crops $\to$ text, Donut maps the entire image directly to a **structured format** (like JSON or Markdown). It is trained to perform sequence generation directly from visual inputs, treating 'reading' as a language translation task where the 'source' is the image pixels.

In [ ]:
class SimpleCrossAttention(nn.Module):
    """
    Conceptual 'Attention' bridge in TrOCR.
    The text decoder looks at visual patches to decide the next character.
    """
    def forward(self, text_queries, visual_patches):
        # text_queries: [B, L, Dim] (current text state)
        # visual_patches: [B, N, Dim] (encoded image patches)
        
        # Simplified Dot-Product Attention
        scores = torch.matmul(text_queries, visual_patches.transpose(-1, -2))
        attn_weights = torch.softmax(scores, dim=-1)
        
        # Weighted sum of patches
        context = torch.matmul(attn_weights, visual_patches)
        return context, attn_weights

# TEST IT
attn = SimpleCrossAttention()
vis = torch.randn(1, 16, 64) # 16 visual patches
txt = torch.randn(1, 5, 64)  # 5 text tokens generated so far

context, weights = attn(txt, vis)
print(f"Attention weights shape: {weights.shape}") # [1, 5, 16] - each token looks at all patches
assert weights.shape == (1, 5, 16)
assert torch.allclose(weights.sum(dim=-1), torch.ones(1, 5))

### COMMON PITFALLS
*   **Perspective Distortion:** OCR models trained on flat documents struggle with 'in-the-wild' images (e.g., street signs). Use **Spatial Transformer Networks (STN)** to rectify the image before recognition.
*   **Aspect Ratio Sensitivity:** CRNNs often resize images to a fixed height (e.g., 32px) but keep the width variable. Fixed-width resizing destroys character shape!
*   **Language Script:** Some scripts (Arabic, Hindi) use 'ligatures' where characters change shape based on neighbors. This requires much higher receptive fields in your RNN/Transformer.

### 🎓 DEEP QUESTION ANSWERED

**Q:** *Why does DBNet's differentiable binarization improve robustness over hard thresholding?*

**A:** In a standard segmentation pipeline, the loss is computed on the probability map. If a signal is 0.49 and the threshold is 0.5, it is 'background'. If it's 0.51, it's 'text'. These hard cut-offs create a non-differentiable landscape. 

By using the **differentiable step function**, the network receives gradients through the binarization process. This allows the model to 'push' the threshold $T$ higher in noisy areas (lowering false positives) and 'pull' it lower in faint text areas (improving recall). This results in much **tighter, more stable bounding boxes** because the post-processing logic is part of the optimization objective.

**Q:** *What happens to CTC Loss if the prediction sequence is shorter than the target?*

**A:** The loss becomes **infinite** (or zero probability). CTC requires that $L_{pred} \ge L_{target} + \text{num\_internal\_repeats}$. If your feature map is too narrow (heavy downsampling), you physically cannot represent the string. This is why text recognition models use small horizontal strides (often stride=1 in later layers).